<a href="https://colab.research.google.com/github/will-mccormack/CS-M148-Proj/blob/main/LAD_Ridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install scikit-lego

In [2]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import time
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklego.linear_model import LADRegression

In [3]:
cleaned_data = pd.read_csv("https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/train.csv")
validation_data = pd.read_csv("https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/validation.csv")

In [4]:
numerical_cols = cleaned_data.select_dtypes(include=np.number).columns.tolist()
predictors = cleaned_data[numerical_cols].drop(columns=["popularity"]).columns.tolist()

model = sm.OLS(cleaned_data["popularity"],cleaned_data[predictors]).fit()
p_values = model.pvalues
p_values_threshold = 0.05
print(p_values >= p_values_threshold)

duration_ms          True
danceability        False
energy              False
key                  True
loudness            False
mode                False
speechiness         False
acousticness         True
instrumentalness    False
liveness            False
valence             False
tempo               False
time_signature      False
dtype: bool


In [5]:
# drop columns with p value over 0.05
drop_columns = p_values[p_values >= p_values_threshold].index.tolist()
cleaned_data = cleaned_data.drop(columns=drop_columns)
validation_data = validation_data.drop(columns=drop_columns)
numerical_cols = cleaned_data.select_dtypes(include=np.number).columns.tolist()
predictors = cleaned_data[numerical_cols].drop(columns=["popularity"]).columns.tolist()

In [6]:
model = sm.OLS(cleaned_data["popularity"],cleaned_data[predictors]).fit()
p_values = model.pvalues
print(p_values >= p_values_threshold)

danceability        False
energy              False
loudness            False
mode                False
speechiness         False
instrumentalness    False
liveness            False
valence             False
tempo               False
time_signature      False
dtype: bool


we can see that there is a good amount of colinearity between some of the values, such as time_signiture, danceability, energy, loudness, and tempo

check the model for training R2 vs validation R2 and MSE

In [7]:
predictions = model.predict(cleaned_data[predictors])
actual_predicted = pd.DataFrame({'Actual Popularity': cleaned_data['popularity'], 'Predicted Popularity': predictions})

px.scatter(actual_predicted, x='Actual Popularity', y='Predicted Popularity')

In [8]:
# Evaluate the model on the validation data
validation_predictions = model.predict(validation_data[predictors])

validation_r2 = r2_score(validation_data['popularity'], validation_predictions)
validation_mse = mean_squared_error(validation_data['popularity'], validation_predictions)

print(f"Validation R-squared: {validation_r2:.4f}")
print(f"Validation MSE: {validation_mse:.4f}")

Validation R-squared: 0.0205
Validation MSE: 486.8454


I want to test if dropping any values seems to give us better predictions

first I will see the p-vlaues and drop the lowest ones

We see here that time signiture has very few unique values, so we should probably drop this. Other columns we should ensure are being properly one hot-encoded for

In [9]:
# Evaluate the model on the training data
train_r2 = model.rsquared
train_mse = mean_squared_error(cleaned_data['popularity'], predictions)

print(f"Training R-squared: {train_r2:.4f}")
print(f"Training MSE: {train_mse:.4f}")

Training R-squared: 0.6997
Training MSE: 488.6674


Training R2 is much higher than validation R2 which means there is high overfitting. We need to apply regularization

In this notebook, the response variable and predictors we are using to model is the popularity and danceability. Using these variables, we can do linear regression.

In [10]:
# training arrays
y_train = cleaned_data["popularity"].values
X_train = cleaned_data.drop(columns=["popularity"])
X_train = pd.get_dummies(X_train, drop_first=False)
X_train = X_train.astype(int).values

In [11]:
# test arrays
y_test = validation_data["popularity"].values
X_test = validation_data.drop(columns=["popularity"])
X_test = pd.get_dummies(X_test, drop_first=False)
X_test = X_test.astype(int).values

Find R^2 for the linear regression model.

In [12]:
#create linear model
regression = LADRegression()

#fit linear model
regression.fit(X_train, y_train)

predicted_y = regression.predict(X_test)

r2 = regression.score(X_test, y_test)
print(f'R^2 = {r2:.5}')

/usr/local/lib/python3.12/dist-packages/sklego/linear_model.py:1358: UserWarning:

Please consider using scikit-learn version of quantile regression.

Hint: `from sklearn.linear_model import QuantileRegressor`
Docs: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.QuantileRegressor.html



R^2 = 0.13863


In [13]:
# train and test shape
y_train.shape, X_train.shape

((61873,), (61873, 124))

Get the regression slope and intercept.


In [ ]:
# R-squared on training data
print(f"Training R^2: {regression.score(X_train, y_train):.4f}")

# MSE on test data
print(f"Test Mean Squared Error (MSE): {mean_squared_error(y_test, predicted_y):.4f}")

# MSE on training data
print(f"Training Mean Squared Error (MSE): {mean_squared_error(y_train, regression.predict(X_train)):.4f}")

# Coefficients and intercept
print("Coefficients: \n", regression.coef_)  # all coefficients
print("Intercept: \n", regression.intercept_)

Training R^2: 0.1691
Test Mean Squared Error (MSE): 428.1216
Training Mean Squared Error (MSE): 414.8600
Coefficients: 
 [ 7.82746108e-01  0.00000000e+00  2.63670447e-01  1.00062485e-02
 -9.46055163e-03  0.00000000e+00  0.00000000e+00 -4.68824923e-02
  0.00000000e+00  3.19518828e-03  3.20553717e+00  1.62349306e+01
 -1.14359770e+01  1.20799666e+01 -2.13348925e+01  1.83096780e+01
  1.76630954e+01 -1.31441814e+01 -8.20905792e+00  2.57467191e+00
  1.31868562e+01 -1.82203116e+01  2.11324098e+01  1.40157090e-01
 -2.27330062e+01  4.16792427e+00  2.47330228e+01 -2.79593007e+01
  5.08875234e+00 -9.91750060e+00 -3.19185458e+01 -2.95646990e+01
 -2.40546635e+00 -8.01232999e+00  1.96559272e+01 -2.40485458e+01
 -1.78314874e-01 -8.64792890e+00 -1.35655423e+01  1.17923363e+01
  1.20969205e+01  2.56551574e+00  4.11061289e+00  1.62107866e+01
  1.83714847e+01  9.99438012e+00  8.03916550e+00  5.41185434e+00
  1.07589339e+01  1.93755971e+00  4.56562634e+00  8.53979020e+00
 -7.68725089e+00 -1.88839561e+01  

Find R^2 and MSE.

Compare rMSE for cross-validation, training, and testing for different polynomial degrees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

np.random.seed(42)

# 2-fold CV for speed (can use 3-5 for more reliable estimate)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

degrees = range(1, 6)  # small degrees for fast test
train_rmse, test_rmse, cv_rmse = [], [], []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())

    # Fit on subset
    model.fit(X_train, y_train)

    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # RMSE
    train_rmse.append(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    test_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred_test)))

    # Cross-validation RMSE
    cv_scores = -cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
    cv_rmse.append(np.sqrt(np.mean(cv_scores)))

# Quick plot
plt.figure(figsize=(6,4))
plt.plot(degrees, train_rmse, 'o-', label='Train RMSE')
plt.plot(degrees, test_rmse, 'o-', label='Test RMSE')
plt.plot(degrees, cv_rmse, 'o-', label='CV RMSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE')
plt.title('Polynomial Regression with CV')
plt.legend()
plt.show()


For linear regression, this might be a sign of underfitting because the training and cross-validation both dramatically decrease in RMSE between degrees 1 and 2.

In [14]:
numeric_cols = cleaned_data.select_dtypes(include=np.number).columns
X = cleaned_data[numeric_cols].drop(columns=['popularity'])
X_std = (X - X.mean()) / X.std()
y = cleaned_data['popularity']

Perform Ridge regression using different hyperparameter values to regularize.

In [15]:
from sklearn.linear_model import  Ridge, Ridge
from sklearn.model_selection import cross_val_score, cross_validate

# use 10-fold cross-validation to select the best lambda (alpha) value for the ridge regression model

# define the alpha values to test
# note that the start/stop values in the first two arguments are the exponents
alphas = np.logspace(-1, 6, 100)

# create an empty list to store the cross-validation scores
ridge_cv_scores = []

# create a for loop to compute the cross-validation score for each alpha value
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge_cv = cross_validate(estimator=ridge,
                              X=X_std,
                              y=y,
                              cv=10,
                              scoring='neg_root_mean_squared_error')
    ridge_cv_scores.append({'alpha': alpha,
                            'log_alpha': np.log(alpha),
                            'test_mse': -np.mean(ridge_cv['test_score'])})

# convert the cross-validation scores into a data frame
ridge_cv_scores_df = pd.DataFrame(ridge_cv_scores)

# plot the cross-validation scores as a function of alpha
px.line(ridge_cv_scores_df,
        x='log_alpha',
        y='test_mse',
        title='Ridge')

Find the hyperparameter that minimizes cross-validation error.

In [16]:
# identify the value of alpha that minimizes the cross-validation score for ridge
ridge_alpha_min = ridge_cv_scores_df.sort_values(by='test_mse').head(1).alpha.values[0]
# compute the min MSE and the SE of the MSE
mse_se_ridge = ridge_cv_scores_df['test_mse'].std() / np.sqrt(10)
mse_min_ridge = ridge_cv_scores_df['test_mse'].min()


# identify the value of alpha that minimizes the cross-validation score for ridge within 1SE
ridge_alpha_1se = ridge_cv_scores_df[(ridge_cv_scores_df['test_mse'] <= mse_min_ridge + mse_se_ridge) &
                                     (ridge_cv_scores_df['test_mse'] >= mse_min_ridge - mse_se_ridge)].sort_values(by='alpha', ascending=False).head(1).alpha.values[0]


In [17]:
print('Ridge (min): ', ridge_alpha_min)
print('Ridge (1SE): ', ridge_alpha_1se)

Ridge (min):  178.8649529057435
Ridge (1SE):  14508.287784959402


Find the R^2 after choosing a hyperparameter.

In [18]:
# Create Ridge models with the optimal alpha values
ridge_min_model = Ridge(alpha=ridge_alpha_min)
ridge_1se_model = Ridge(alpha=ridge_alpha_1se)

# Standardize the test data using the mean and std from the training data
test_numeric = validation_data.select_dtypes(include=np.number)
test_numeric = test_numeric.drop(columns=['popularity'])
test_numeric_aligned = test_numeric.reindex(columns=X.columns, fill_value=0)
X_test_std = (test_numeric_aligned - X.mean()) / X.std()
y_test = validation_data['popularity']

# Fit the models on the training data (using standardized X_std)
ridge_min_model.fit(X_std, y)
ridge_1se_model.fit(X_std, y)

# Make predictions on the standardized test data
ridge_min_pred = ridge_min_model.predict(X_test_std)
ridge_1se_pred = ridge_1se_model.predict(X_test_std)

ridge_min_rmse = np.sqrt(mean_squared_error(y_test, ridge_min_pred))
ridge_1se_rmse = np.sqrt(mean_squared_error(y_test, ridge_1se_pred))

ridge_min_r2 = r2_score(y_test, ridge_min_pred)
ridge_1se_r2 = r2_score(y_test, ridge_1se_pred)

print("Ridge (min alpha) - RMSE:", ridge_min_rmse, "R-squared:", ridge_min_r2)
print("Ridge (1SE alpha) - RMSE:", ridge_1se_rmse, "R-squared:", ridge_1se_r2)

Ridge (min alpha) - RMSE: 22.003531330451683 R-squared: 0.02589281517339137
Ridge (1SE alpha) - RMSE: 22.02083536820938 R-squared: 0.024360096156266886


R-squared for ridge is greater than R-squared for linear regression, which indicates that it's doing better than before ridge.

I want to check if dropping any columns can improve R2, because our current model has a very low R2 and can't explain much. I want to calculate colinearity, to maybe slim the model down a little as a start:

In [19]:
X = cleaned_data[predictors]
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

            feature        VIF
0      danceability  17.681508
1            energy  24.439614
2          loudness  12.376596
3              mode   2.785675
4       speechiness   1.876563
5  instrumentalness   1.717592
6          liveness   2.504694
7           valence   6.657111
8             tempo  20.056534
9    time_signature  82.367140


We see an incredibly high time_signiture VIF value, which seems odd. Worth checking out unique values, because if it is a bunch of the same values it could muddy our model.

In [20]:
print(p_values)
cleaned_data['time_signature'].value_counts()

danceability         1.016408e-53
energy               2.714522e-07
loudness             2.087274e-07
mode                 2.143193e-05
speechiness          1.234114e-58
instrumentalness    4.005817e-154
liveness             4.073999e-04
valence             6.266045e-138
tempo                8.673295e-16
time_signature       0.000000e+00
dtype: float64


,count
time_signature,
4,60840
5,1033


Most of these have the same value, lets try a model dropping this. While were at it, let's try models dropping other colinear values:

In [21]:

base_features = ['danceability', 'energy', 'loudness', 'mode',
                 'speechiness', 'instrumentalness', 'liveness',
                 'valence', 'tempo', 'time_signature']

for drop in [ ['energy'],
              ['time_signature'],
             ['loudness'],
             ['danceability'],
             ['energy', 'loudness'],
             ['energy', 'loudness', 'tempo'],
             ['time_signature', 'energy', 'loudness', 'tempo']]:

    features = [f for f in base_features if f not in drop]
    model = sm.OLS(cleaned_data["popularity"], cleaned_data[features]).fit()
    print(drop, model.rsquared)

['energy'] 0.6996078843275984
['time_signature'] 0.6899138774137228
['loudness'] 0.6996054189829803
['danceability'] 0.6985783155534432
['energy', 'loudness'] 0.699588979001911
['energy', 'loudness', 'tempo'] 0.6992895147139675
['time_signature', 'energy', 'loudness', 'tempo'] 0.6586874468500042


These are all huge improvements in prediction power compared to our earlier models---for some reason, dropping these columns improves things a lot. Is it colinearity that is the cause of this? Or outliers in these categories greatly skewing things? We should test R2 against validation data like we did on our model before to see if this change can propogate to validation set as well